# **Instrument to Instrument (ITI) translation: Training**

## Read The Docs
For more information about the tool and background of the individual case studies see:

[iti-documentation.rtfd.io](https://iti-documentation.readthedocs.io/en/latest/)

ITI provides translations between image domains of solar instruments (image enhancement, super-resolution, cross-calibration, estimation of observables). This notebook provides two training examples of ITI.

Colab offers free online computation power. The training requires an active GPU. This can be changed in the menu (Runtime -> Change runtime type -> Hardware accelerator -> GPU).

## Installation

In [1]:
!pip install git+https://github.com/spaceml-org/InstrumentToInstrument.git@development
!pip install lightning
!pip install sunpy_soar
!pip install sunpy

  Cloning https://github.com/spaceml-org/InstrumentToInstrument.git (to revision development) to /private/var/folders/tj/7h3lqn950sv5k36g1yyl4vkr0000gn/T/pip-req-build-l125clgz
  Running command git clone --filter=blob:none --quiet https://github.com/spaceml-org/InstrumentToInstrument.git /private/var/folders/tj/7h3lqn950sv5k36g1yyl4vkr0000gn/T/pip-req-build-l125clgz
  Running command git checkout -b development --track origin/development
  Switched to a new branch 'development'
  Branch 'development' set up to track remote branch 'development' from 'origin'.
  Resolved https://github.com/spaceml-org/InstrumentToInstrument.git to commit b083c705d463997caebf2df663d8774cbe569eda
  Preparing metadata (setup.py) ... done

[notice] A new release of pip is available: 23.2.1 -> 24.1.2
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 23.2.1 -> 24.1.2
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 23

In [2]:
import gcsfs
import zarr
import dask.array as da

import glob
import os
import logging

import numpy as np

from torch.utils.data import Dataset, DataLoader
from multiprocessing import get_context

from sdo.datasets.sdo_dataset import SDO_Dataset
from sdo.pytorch_utilities import create_dataloader

from iti.data.editor import *
from iti.data.dataset import BaseDataset, StackDataset, sdo_norms
from iti.train.model import DiscriminatorMode
from iti.trainer import Trainer, loop
from iti.evaluation.util import download_gcp_bucket
from urllib.request import urlretrieve

from matplotlib import pyplot as plt
from matplotlib.colors import Normalize
from matplotlib.colors import LogNorm
from astropy.visualization import ImageNormalize, LinearStretch, AsinhStretch

from sunpy.visualization.colormaps import cm

from datetime import datetime, timedelta

import pandas as pd

from astropy import units as u
from astropy.coordinates import SkyCoord
import warnings
warnings.filterwarnings('ignore')
os.makedirs('data', exist_ok=True)

from tqdm import tqdm

import torch
base_path = os.getcwd()

## Download SDOML Dataset

The SDOML dataset is publicly available for download. The data is stored as compressed Numpy arrays.

For this demo we use observations from 2011, where we select one observation per day. For practical applications all avaialable years should be considered (2010 - 2021).

We use the HMI magnetograms as reference and select the corresponding EUV observations.

In [3]:
gcs = gcsfs.GCSFileSystem(access='read_only')

In [4]:
loc_hmi = "iti-sdomlv2a-2011/HMI.zarr/2011"
store = gcsfs.GCSMap(loc_hmi, gcs=gcs, check=False)
root = zarr.group(store)

In [5]:
print(root.tree())

/
 ├── Bx (40154, 512, 512) float32
 ├── By (40154, 512, 512) float32
 └── Bz (40154, 512, 512) float32


In [11]:
hmi_times = root["Bx"].attrs["T_OBS"]
sampling_step = len(hmi_times) // 365
hmi_times = hmi_times[::sampling_step] # subsample
hmi_times = pd.to_datetime(hmi_times, format='%Y.%m.%d_%H:%M:%S.%f_TAI').to_pydatetime()

hmi_Bx = da.from_array(root["Bx"])[::sampling_step] # subsample
hmi_By = da.from_array(root["By"])[::sampling_step] # subsample
hmi_Bz = da.from_array(root["Bz"])[::sampling_step] # subsample

In [12]:
loc = "iti-sdomlv2a-2011/AIA.zarr/2011"
store = gcsfs.GCSMap(loc, gcs=gcs, check=False)
aia_root = zarr.group(store, synchronizer=zarr.ThreadSynchronizer())

In [13]:
# align AIA data
aia_keys = ['171A', '193A', '211A', '304A']# all keys: aia_root.array_keys()
time_data_mapping = {t: [] for t in hmi_times}
for key in aia_keys: 
  df = aia_root[key]
  obs_times = pd.to_datetime(df.attrs['T_OBS'], format='%Y-%m-%dT%H:%M:%S.%fZ').to_pydatetime()
  da_array = da.from_array(df)
  for t in hmi_times:
    if np.min(np.abs(obs_times - t)) > timedelta(minutes=15):
      continue
    idx = np.argmin(np.abs(obs_times - t))
    time_data_mapping[t] += [da_array[idx]]

In [ ]:
for i, (d, aia_cube) in tqdm(enumerate(time_data_mapping.items()), desc='loading data cubes', total=len(hmi_times)):
  if len(aia_cube) != len(aia_keys):
    continue
  save_path = 'data/%s.npy' % d.isoformat('T')
  if os.path.exists(save_path):
    continue
  # subsample to 256x256
  cube = np.stack([hmi_Bx[i, ::2, ::2], 
                   hmi_By[i, ::2, ::2], 
                   hmi_Bz[i, ::2, ::2], 
                   *[d[::2, ::2] for d in aia_cube]])
  np.save(save_path, cube)

loading data cubes:   0%|          | 0/366 [00:00<?, ?it/s]